In [1]:
import sys; sys.path.append("..")
import pandas as pd
from src.config import CONTAMINANTES
from src.limpieza import limpiar_negativos, limpiar_saturados, limpiar_inconsistentes, reindexar_gaps

In [2]:
datos = pd.read_parquet("../data/crudo.parquet")
print(f"Crudo: {len(datos):,} filas")

Crudo: 7,999,805 filas


In [3]:
ancho = datos.pivot_table(
    index=["timestamp", "estacion", "comuna"],
    columns="parametro",
    values="valor",
    aggfunc="mean"
).reset_index()
ancho.columns.name = None
print(f"Ancho: {ancho.shape[0]:,} filas x {ancho.shape[1]} columnas")

Ancho: 1,643,491 filas x 11 columnas


In [4]:
ancho["anio"] = ancho["timestamp"].dt.year
df = ancho[ancho["anio"] >= 2017].copy()
print(f"2017+: {len(df):,} filas")

2017+: 1,124,194 filas


In [5]:
contaminantes = [c for c in CONTAMINANTES.keys() if c in df.columns]
df = limpiar_negativos(df, contaminantes)
df = limpiar_saturados(df, contaminantes)
df = limpiar_inconsistentes(df)
df = reindexar_gaps(df)
print(f"\nFinal: {len(df):,} filas")

REPORTE DE LIMPIEZA DE NEGATIVOS
---------------------------------------------
  mp2.5  | ruido → 0:     11 | falla → NaN:   5
  mp10   | ruido → 0:      1 | falla → NaN:   0
  so2    | ruido → 0:  2,249 | falla → NaN:   0
  no     | ruido → 0:      0 | falla → NaN:   0
  no2    | ruido → 0:    369 | falla → NaN:   0
  co     | ruido → 0:  3,139 | falla → NaN:   0
  o3     | ruido → 0:     54 | falla → NaN:   0
  nox    | ruido → 0:      0 | falla → NaN:   0

REPORTE DE LIMPIEZA DE SATURADOS
---------------------------------------------
  mp2.5  | saturado → NaN:   5
          timestamp                estacion  mp2.5
2018-05-24 11:00:00                 Hualqui 1996.0
2019-05-13 14:00:00                Punteras 2000.0
2019-05-13 16:00:00                Punteras 2000.0
2024-07-09 18:00:00 Consultorio San Vicente 2000.0
2024-07-09 19:00:00 Consultorio San Vicente 2000.0
  mp10   | saturado → NaN:   5
          timestamp                estacion   mp10
2019-05-13 14:00:00                Pun

In [6]:
df.to_parquet("../data/dataset_limpio.parquet", index=False)
print("dataset_limpio.parquet guardado ✅")

dataset_limpio.parquet guardado ✅
